In [14]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from ddgs import DDGS


# =========================================================
# 1. Model
# =========================================================

model = init_chat_model(
    "llama3.1:8b",
    model_provider="ollama",
    temperature=0,
)


# =========================================================
# 2. Web Search Tool
# =========================================================

@tool
def web_search(query: str) -> str:
    """
    جستجوی اطلاعات در اینترنت.

    زمانی از این ابزار استفاده کن که سؤال کاربر
    به اطلاعات جدید، اخبار، نسخه‌های نرم‌افزار،
    قیمت‌ها یا اطلاعات موجود در اینترنت نیاز دارد.
    """

    print("\n" + "=" * 60)
    print("[TOOL] web_search")
    print(f"[QUERY] {query}")
    print("=" * 60)

    try:
        # انجام جستجو
        results = DDGS().text(
            query,
            region="ir-fa",
            safesearch="moderate",
            max_results=5,
        )

    except Exception as e:
        print("[TOOL ERROR]")
        print(e)

        return f"خطا در جستجوی اینترنت: {e}"

    # اگر نتیجه‌ای پیدا نشد
    if not results:
        print("[TOOL RESULT] No results")

        return "هیچ نتیجه‌ای برای این جستجو پیدا نشد."

    # تبدیل نتایج به متن قابل فهم برای LLM
    formatted_results = []

    for i, result in enumerate(results, start=1):

        title = result.get("title", "")
        body = result.get("body", "")
        url = result.get("href", "")

        formatted_results.append(
            f"""
نتیجه شماره {i}

عنوان:
{title}

خلاصه:
{body}

URL:
{url}
"""
        )

    final_result = "\n".join(formatted_results)

    print(f"[TOOL RESULT] {len(results)} results found")

    return final_result


# =========================================================
# 3. System Prompt
# =========================================================

SYSTEM_PROMPT = """
تو یک دستیار تحقیقاتی دقیق و قابل اعتماد هستی.

وظایف تو:

1. سؤال کاربر را به دقت تحلیل کن.

2. اگر پاسخ سؤال به اطلاعات به‌روز یا اطلاعات موجود
   در اینترنت نیاز دارد، حتماً از ابزار web_search استفاده کن.

3. برای موضوعاتی مانند:
   - اخبار
   - آخرین نسخه نرم‌افزارها
   - قیمت‌ها
   - تغییرات اخیر فناوری
   - رویدادهای جدید
   - اطلاعاتی که ممکن است تغییر کرده باشند

   از web_search استفاده کن.

4. اگر برای پاسخ به سؤال به اینترنت نیاز نیست،
   بدون استفاده از ابزار پاسخ بده.

5. اگر از web_search استفاده کردی:
   - نتایج را تحلیل کن.
   - اطلاعات متناقض را تا حد امکان تشخیص بده.
   - اطلاعات نامطمئن را به عنوان حقیقت قطعی بیان نکن.
   - URL منابع مهم را در پاسخ نهایی ذکر کن.

6. نتایج خام ابزار را مستقیماً به کاربر تحویل نده.
   ابتدا آنها را تحلیل و خلاصه کن.

7. پاسخ نهایی را به زبان فارسی بده.

8. پاسخ را واضح، دقیق و نسبتاً کوتاه نگه دار.
"""


# =========================================================
# 4. Create Agent
# =========================================================

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    name="web_research_agent",
)


# =========================================================
# 5. Run Agent
# =========================================================

def ask_agent(question: str):

    print("\n")
    print("#" * 70)
    print("USER QUESTION")
    print("#" * 70)

    print(question)

    print("\n")
    print("#" * 70)
    print("AGENT STARTED")
    print("#" * 70)

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    # =====================================================
    # نمایش مسیر اجرای Agent
    # =====================================================

    print("\n")
    print("#" * 70)
    print("AGENT EXECUTION DETAIL:")
    print("#" * 70)

    for message in result["messages"]:

        print("\n" + "-" * 60)

        print(
            "MESSAGE TYPE:",
            type(message).__name__
        )

        # اگر مدل Tool Call ساخته باشد
        if hasattr(message, "tool_calls"):

            if message.tool_calls:

                print("\nTOOL CALLS:")

                for tool_call in message.tool_calls:

                    print(
                        f"  Tool: {tool_call['name']}"
                    )

                    print(
                        f"  Args: {tool_call['args']}"
                    )

        # نمایش محتوای پیام
        if message.content:

            print("\nCONTENT:")

            print(message.content)

    # =====================================================
    # Final Answer
    # =====================================================

    final_message = result["messages"][-1]

    print("\n")
    print("#" * 70)
    print("FINAL ANSWER:")
    print("#" * 70)

    print(final_message.content)

    return final_message.content


# =============================================
ask_agent("آخرین اخبار عربستان رو بگو")



######################################################################
USER QUESTION
######################################################################
آخرین اخبار عربستان رو بگو


######################################################################
AGENT STARTED
######################################################################

[TOOL] web_search
[QUERY] آخرین اخبار عربستان
[TOOL RESULT] 5 results found


######################################################################
AGENT EXECUTION DETAIL:
######################################################################

------------------------------------------------------------
MESSAGE TYPE: HumanMessage

CONTENT:
آخرین اخبار عربستان رو بگو

------------------------------------------------------------
MESSAGE TYPE: AIMessage

TOOL CALLS:
  Tool: web_search
  Args: {'query': 'آخرین اخبار عربستان'}

------------------------------------------------------------
MESSAGE TYPE: ToolMessage

CONTENT:

نتیجه شماره 1

عنوان:
جدیدتر

'آخرین اخبار عربستان به این شرح است:\n\n* وزیر امور خارجه عربستان تماسی از همتای اماراتی خود دریافت کرده است.\n* عربستان سعودی در حال تلاش برای پایان دادن به محاصره و تجاوز در یمن است.\n* فرمانده سنتکام در جده عربستان با ولیعهد سعودی دیدار کرده است.\n* عربستان سعودی در حال الگوگیری از آمریکا است.\n* بنسلمان فردا به مصر میرود و امنیت دریای سرخ و تحولات منطقه در دستور کار است.\n* تیم ملی عربستان در حال تلاش برای قهرمانی آسیاست.\n\nمنبع: نتایج جستجوی وب'